In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import json
from IPython.display import display


In [ ]:
def momentum_screener(tickers, period='1y', threshold_json='thresholds.json'):
    """
    Complete momentum analysis from scratch - fetches data, calculates metrics, and applies threshold-based coloring.
    
    Parameters:
    -----------
    tickers : list
        List of ticker symbols to analyze
    period : str
        Data period to fetch (default: '1y'). Options: '6mo', '1y', '2y', '5y'
    threshold_json : str
        Path to JSON file containing threshold definitions
    
    Returns:
    --------
    DataFrame with calculated metrics and styled output
    """
    
    # Load thresholds from JSON
    with open(threshold_json, 'r') as f:
        thresholds = json.load(f)
        
    # Fetch benchmark data (VOO) for RS comparison
    print("Fetching benchmark data (VOO)...")
    voo_close = None
    try:
        voo_data = yf.download('VOO', period=period, progress=False)
        
        # Handle MultiIndex for VOO
        if isinstance(voo_data.columns, pd.MultiIndex):
            try:
                if 'VOO' in voo_data.columns.get_level_values(1):
                    voo_data = voo_data.xs('VOO', axis=1, level=1)
                else:
                    voo_data.columns = voo_data.columns.droplevel(1)
            except:
                pass
        
        # Extract Close column
        if 'Close' in voo_data.columns:
             temp_close = voo_data['Close']
             if isinstance(temp_close, pd.DataFrame):
                 voo_close = temp_close.iloc[:, 0]
             else:
                 voo_close = temp_close
    except Exception as e:
        print(f"  Warning: Failed to download VOO data: {e}")
    
    results = []
    
    from sklearn.linear_model import LinearRegression
    
    for ticker in tickers:
        try:
            print(f"Processing {ticker}...")
            
            # Download data
            data = yf.download(ticker, period=period, progress=False)
            
            if data.empty or len(data) < 200:
                print(f"  Insufficient data for {ticker}")
                continue
            
            # Handle MultiIndex columns (common in new yfinance)
            if isinstance(data.columns, pd.MultiIndex):
                try:
                    if ticker in data.columns.get_level_values(1):
                         data = data.xs(ticker, axis=1, level=1)
                    else:
                         data.columns = data.columns.droplevel(1)
                except Exception as e:
                     print(f"  Warning: Could not handle MultiIndex for {ticker}: {e}")
            
            if isinstance(data, pd.DataFrame) and 'Close' in data.columns:
                close_col = data['Close']
                if isinstance(close_col, pd.DataFrame):
                    print(f"  Warning: Duplicate 'Close' columns found for {ticker}, using first one.")
                    data = data.loc[:, ~data.columns.duplicated()]
            
            current_price = data['Close'].iloc[-1]
            returns = data['Close'].pct_change()
            
            # Calculate ratio series (ticker/VOO)
            ratio_series = None
            if voo_close is not None:
                try:
                    t_series = data['Close']
                    if isinstance(t_series, pd.DataFrame):
                        t_series = t_series.iloc[:, 0]
                    v_series = voo_close
                    if isinstance(v_series, pd.DataFrame):
                        v_series = v_series.iloc[:, 0]
                    aligned = pd.concat([t_series, v_series], axis=1, join='inner').dropna()
                    ratio_series = aligned.iloc[:, 0] / aligned.iloc[:, 1]
                except Exception:
                    ratio_series = None
            
            # RS vs VOO (6M)
            rs_vs_voo = np.nan
            if ratio_series is not None and len(ratio_series) >= 126:
                try:
                    rs_vs_voo = ((ratio_series.iloc[-1] / ratio_series.shift(126).iloc[-1]) - 1) * 100
                except Exception as e:
                    pass
            
            # Moving averages
            if ratio_series is not None and len(ratio_series) >= 200:
                sma_50 = ratio_series.rolling(50, min_periods=50).mean().iloc[-1]
                sma_200_series = ratio_series.rolling(200, min_periods=200).mean()
                sma_200 = sma_200_series.iloc[-1]
            else:
                sma_50 = np.nan
                sma_200 = np.nan
                sma_200_series = None
            
            # Ratio vs ratio SMA metrics
            if ratio_series is not None and np.isfinite(sma_50) and np.isfinite(sma_200):
                current_ratio = ratio_series.iloc[-1]
                sma_50_val = ratio_series.rolling(50, min_periods=50).mean().iloc[-1]
                sma_200_val = sma_200
                ratio_vs_sma50  = (current_ratio / sma_50_val  - 1) * 100
                ratio_vs_sma200 = (current_ratio / sma_200_val - 1) * 100
                sma50_vs_sma200 = (sma_50_val / sma_200_val - 1) * 100
                ratio_above_sma200 = current_ratio > sma_200_val
                sma50_above_sma200 = sma_50_val > sma_200_val
                # Ratio SMA200 slope (50d)
                if sma_200_series is not None and len(sma_200_series.dropna()) >= 50:
                    y = sma_200_series.iloc[-50:].values.reshape(-1,1)
                    x = np.arange(len(y)).reshape(-1,1)
                    model = LinearRegression().fit(x, y)
                    slope = model.coef_[0][0]
                    slope_pct = slope / sma_200_series.iloc[-1] * 100 if sma_200_series.iloc[-1] != 0 else np.nan
                    ratio_sma200_slope_50d = slope_pct
                else:
                    ratio_sma200_slope_50d = np.nan
                # OBV calculation
                obv = None
                if 'Volume' in data.columns:
                    volume = data['Volume']
                    price = data['Close']
                    direction = np.sign(price.diff().fillna(0))
                    obv = (direction * volume).cumsum()
                # Ratio OBV vs OBV SMA50
                ratio_obv_vs_obv_sma50 = np.nan
                if obv is not None and len(obv) >= 50:
                    obv_sma50 = obv.rolling(50, min_periods=50).mean()
                    if np.isfinite(obv.iloc[-1]) and np.isfinite(obv_sma50.iloc[-1]) and obv_sma50.iloc[-1] != 0:
                        ratio_obv_vs_obv_sma50 = obv.iloc[-1] / obv_sma50.iloc[-1]
                # Ratio OBV ROC (20d, %)
                ratio_obv_roc_20d = np.nan
                if obv is not None and len(obv) >= 20:
                    try:
                        ratio_obv_roc_20d = ((obv.iloc[-1] / obv.iloc[-21]) - 1) * 100
                    except Exception:
                        ratio_obv_roc_20d = np.nan
                # Max Ratio Drawdown (6mo, %)
                max_ratio_drawdown_6mo = np.nan
                if ratio_series is not None and len(ratio_series) >= 126:
                    window = 126
                    rolling_max = ratio_series.rolling(window, min_periods=window).max()
                    drawdown = (ratio_series - rolling_max) / rolling_max * 100
                    max_ratio_drawdown_6mo = drawdown.min()
            else:
                ratio_vs_sma50 = ratio_vs_sma200 = sma50_vs_sma200 = np.nan
                ratio_above_sma200 = sma50_above_sma200 = False
                ratio_sma200_slope_50d = np.nan
                ratio_obv_vs_obv_sma50 = np.nan
                ratio_obv_roc_20d = np.nan
                max_ratio_drawdown_6mo = np.nan
            
            # Days ratio above ratio SMA50 percentage (use .shift for robustness)
            if ratio_series is not None and len(ratio_series) >= 50:
                sma50_series = ratio_series.rolling(50, min_periods=50).mean()
                valid = sma50_series.notna()
                days_above_sma50_pct = (ratio_series[valid] > sma50_series[valid]).mean() * 100
            else:
                days_above_sma50_pct = np.nan
            
            # RSI calculation
            rsi_current = np.nan
            rsi_series = None
            if ratio_series is not None and len(ratio_series) >= 15:
                ratio_delta = ratio_series - ratio_series.shift(1)
                gain = ratio_delta.where(ratio_delta > 0, 0)
                loss = -ratio_delta.where(ratio_delta < 0, 0)
                avg_gain = gain.rolling(window=14, min_periods=14).mean()
                avg_loss = loss.rolling(window=14, min_periods=14).mean()
                rs = avg_gain / avg_loss
                rsi = 100 - (100 / (1 + rs))
                rsi_current = rsi.iloc[-1]
                rsi_series = rsi
            
            # RSI Up/Down Volatility Ratio
            rsi_volatility_ratio = np.nan
            if rsi_series is not None and len(rsi_series) > 15:
                rsi_diff = rsi_series.diff()
                up_vol = rsi_diff[rsi_diff > 0].std()
                down_vol = rsi_diff[rsi_diff < 0].std()
                if down_vol is not None and down_vol != 0 and not np.isnan(down_vol):
                    rsi_volatility_ratio = up_vol / abs(down_vol)
                else:
                    rsi_volatility_ratio = np.nan
            
            # Volatility (annualized, using ratio series)
            volatility = np.nan
            if ratio_series is not None and len(ratio_series) > 1:
                ratio_returns = ratio_series.pct_change().dropna()
                if len(ratio_returns) > 1:
                    volatility = ratio_returns.std() * np.sqrt(252) * 100
            
            # MACD calculation
            if ratio_series is not None and len(ratio_series) >= 26:
                ema_12 = ratio_series.ewm(span=12, adjust=False).mean()
                ema_26 = ratio_series.ewm(span=26, adjust=False).mean()
                macd = ema_12 - ema_26
                macd_signal = macd.ewm(span=9, adjust=False).mean()
                macd_hist = macd - macd_signal
                macd_norm = (macd.iloc[-1] / ema_26.iloc[-1]) * 100
                macd_hist_norm = (macd_hist.iloc[-1] / ema_26.iloc[-1]) * 100
                if len(macd_hist) >= 60:
                    macd_hist_change = ((macd_hist.iloc[-1] / ema_26.iloc[-1]) - 
                                       (macd_hist.iloc[-60] / ema_26.iloc[-60])) * 100
                else:
                    macd_hist_change = np.nan
            else:
                macd_norm = np.nan
                macd_hist_norm = np.nan
                macd_hist_change = np.nan
            
            def to_scalar(val):
                if isinstance(val, (pd.Series, pd.DataFrame, np.ndarray, list)):
                    try:
                         if hasattr(val, 'item'):
                             return val.item()
                         if len(val) == 1:
                             return val[0]
                    except:
                        pass
                    return np.nan
                return val
            
            result = {
                'Ticker': ticker,
                'Ratio RS vs VOO (6M, ratio, %)': to_scalar(rs_vs_voo),
                'Price': to_scalar(current_price),
                'Ratio vs SMA200 (ratio, %)': to_scalar(ratio_vs_sma200),
                'Ratio SMA200 Slope (50d, %)': to_scalar(ratio_sma200_slope_50d),
                'Ratio OBV vs OBV SMA50': to_scalar(ratio_obv_vs_obv_sma50),
                'Ratio OBV ROC (20d, %)': to_scalar(ratio_obv_roc_20d),
                'Max Ratio Drawdown (6mo, %)': to_scalar(max_ratio_drawdown_6mo),
                'Days Above SMA50 (ratio, %)': to_scalar(days_above_sma50_pct),
                'RSI (14, ratio)': to_scalar(rsi_current),
                'RSI Up/Down Volatility Ratio': to_scalar(rsi_volatility_ratio),
                'Volatility (ratio, %)': to_scalar(volatility),
                'MACD (ratio, %)': to_scalar(macd_norm),
                'MACD Histogram (ratio, %)': to_scalar(macd_hist_norm),
                'MACD Hist Change (60d, ratio, %)': to_scalar(macd_hist_change),
            }
            results.append(result)
        except Exception as e:
            print(f"  Error processing {ticker}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    df = pd.DataFrame(results)
    
    if df.empty:
        print("No valid results to display")
        return None
    
    df.set_index('Ticker', inplace=True)
    
    def get_band_index(val, threshold_edges):
        if pd.isna(val):
            return -1
        for i, edge in enumerate(threshold_edges):
            if val < edge:
                return i
        return len(threshold_edges)
    
    display_cols = [col for col in df.columns if col in thresholds]
    df_display = df[display_cols].copy()
    
    def apply_column_styles(series):
        col_name = series.name
        if col_name not in thresholds:
            return [''] * len(series)
        threshold_config = thresholds[col_name]
        edges = threshold_config['thresholds']
        colors = threshold_config['colors']
        text_colors = threshold_config['text_colors']
        styles = []
        for val in series:
            if isinstance(val, (pd.Series, np.ndarray, list)):
                 styles.append('background-color: #ff0000; color: #ffffff')
                 continue
            if pd.isna(val):
                styles.append('background-color: #d3d3d3; color: #000000')
                continue
            band_idx = get_band_index(val, edges)
            if band_idx >= len(colors):
                band_idx = len(colors) - 1
            bg_color = colors[band_idx]
            txt_color = text_colors[band_idx]
            styles.append(f'background-color: {bg_color}; color: {txt_color}')
        return styles
    
    styled = df_display.style.apply(apply_column_styles, axis=0)
    
    format_dict = {}
    for col in display_cols:
        if 'volatility' in col.lower():
            format_dict[col] = '{:.2f}'
        elif 'rsi' in col.lower():
            format_dict[col] = '{:.1f}'
        else:
            format_dict[col] = '{:.2f}'
    styled = styled.format(format_dict, na_rep='-')
    display(styled)
    return df


In [26]:
# Example usage
tickers = ['VOO', 'IAU', 'SIVR', 'XAR', 'GOOG', 'AVGO', 'HSBC', 'JNJ', 'TEL', 'VLO']
df = momentum_screener(tickers, period='1y')



Fetching benchmark data (VOO)...
Processing VOO...
Processing IAU...
Processing SIVR...
Processing XAR...
Processing GOOG...
Processing AVGO...
Processing HSBC...
Processing JNJ...
Processing TEL...
Processing VLO...


,"RS vs VOO (6M, ratio, %)","Price vs SMA200 (ratio, %)","Ratio SMA200 Slope (50d, %)",Ratio OBV vs OBV SMA50,"Ratio OBV ROC (20d, %)","Max Ratio Drawdown (6mo, %)","Days Above SMA50 (ratio, %)","RSI (14, ratio)",RSI Up/Down Volatility Ratio,"Volatility (ratio, %)","MACD (ratio, %)","MACD Histogram (ratio, %)","MACD Hist Change (60d, ratio, %)"
Ticker,,,,,,,,,,,,,
VOO,0.00,-0.00,0.00,1.33,78.96,-0.00,18.81,-,0.60,0.00,-0.00,0.00,-0.00
IAU,23.24,13.29,0.11,1.21,6.96,-22.97,62.38,53.5,1.03,26.56,1.52,0.20,0.17
SIVR,111.12,82.51,0.26,1.61,62.80,-16.73,80.20,61.5,1.00,35.60,9.27,1.31,2.05
XAR,19.06,24.08,0.10,2.14,158.12,-11.32,73.27,91.2,0.91,17.67,4.75,1.30,1.72
GOOG,60.47,34.21,0.21,1.12,26.08,-12.83,75.74,69.7,1.04,24.99,1.76,0.27,0.33
AVGO,12.35,9.07,0.19,1.32,95.34,-19.81,76.24,49.2,1.08,43.42,-1.57,0.32,0.49
HSBC,20.22,16.22,0.08,1.17,19.69,-9.33,73.76,62.8,1.10,20.35,2.15,-0.06,0.58
JNJ,22.12,14.22,0.06,1.09,-0.48,-14.18,65.84,66.6,1.18,26.07,1.05,0.53,0.48
TEL,21.88,12.44,0.14,1.11,6.82,-10.57,75.74,64.0,0.92,20.73,0.28,0.53,0.33
